In [2]:
%pip install pandas pymysql sqlalchemy ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine

# --- Database Configurations ---
DB_USER = "root"
DB_PASSWORD = "Nihaz@123"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "olist_db"
DATA_DIR = "../data_raw"

# URL-encode password to safely handle special characters
safe_password = urllib.parse.quote_plus(DB_PASSWORD)

# Initialize SQLAlchemy engine for MySQL
connection_string = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)
print("Connected to MySQL successfully.")

# Bulk ingest raw CSV data into database
for file in os.listdir(DATA_DIR):
    if file.endswith(".csv"):
        table_name = file.replace(".csv", "")
        file_path = os.path.join(DATA_DIR, file)
        
        print(f"Loading {file} into MySQL table '{table_name}'...")
        
        df = pd.read_csv(file_path)
        
        # Batch upload to manage memory footprint for dense datasets
        df.to_sql(table_name, con=engine, if_exists="replace", index=False, chunksize=10000)

print("\nAll datasets loaded successfully into the database!")

Connected to MySQL successfully.
Loading clean_customer_clv.csv into MySQL table 'clean_customer_clv'...
Loading clean_products.csv into MySQL table 'clean_products'...
Loading olist_customers_dataset.csv into MySQL table 'olist_customers_dataset'...
Loading olist_geolocation_dataset.csv into MySQL table 'olist_geolocation_dataset'...
Loading olist_orders_dataset.csv into MySQL table 'olist_orders_dataset'...
Loading olist_order_items_dataset.csv into MySQL table 'olist_order_items_dataset'...
Loading olist_order_payments_dataset.csv into MySQL table 'olist_order_payments_dataset'...
Loading olist_order_reviews_dataset.csv into MySQL table 'olist_order_reviews_dataset'...
Loading olist_products_dataset.csv into MySQL table 'olist_products_dataset'...
Loading olist_sellers_dataset.csv into MySQL table 'olist_sellers_dataset'...
Loading product_category_name_translation.csv into MySQL table 'product_category_name_translation'...

All datasets loaded successfully into the database!


In [4]:
# Load raw product and translation reference tables
df_products = pd.read_sql("SELECT * FROM olist_products_dataset", engine)
df_translation = pd.read_sql("SELECT * FROM product_category_name_translation", engine)

# Count initial unmapped category records
missing_count = df_products["product_category_name"].isnull().sum()
print(f"Found {missing_count} products missing a category name.")

# Standardize missing values in native category field
df_products["product_category_name"] = df_products["product_category_name"].fillna("unclassified")
df_products["product_category_name"] = df_products["product_category_name"].replace("", "unclassified")

# Map Portuguese category names to English translations
df_clean_products = pd.merge(df_products, df_translation, on="product_category_name", how="left")

# Assign default string value to unmapped English values
df_clean_products["product_category_name_english"] = df_clean_products["product_category_name_english"].fillna("Unclassified/Unknown")

# Export clean products dimension table for Power BI ingestion
df_clean_products.to_csv("../data_raw/clean_products.csv", index=False)

print("Processing complete! 'clean_products.csv' saved safely inside data_raw folder.")
df_clean_products[["product_id", "product_category_name", "product_category_name_english"]].head()

Found 610 products missing a category name.
Processing complete! 'clean_products.csv' saved safely inside data_raw folder.


,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [5]:
print("Processing Advanced 4-Tier CLV Lifecycle Engine...")

# Load transaction data tables
df_orders = pd.read_sql("SELECT order_id, customer_id, order_status FROM olist_orders_dataset", engine)
df_customers = pd.read_sql("SELECT customer_id, customer_unique_id FROM olist_customers_dataset", engine)
df_payments = pd.read_sql("SELECT order_id, payment_value FROM olist_order_payments_dataset", engine)

# Merge transactional logs across relational keys
df_order_payments = pd.merge(df_orders, df_payments, on="order_id", how="left")
df_customer_spending = pd.merge(df_order_payments, df_customers, on="customer_id", how="left")

# Filter dataset to capture active operational order statuses
valid_statuses = ["delivered", "shipped", "approved", "processing", "invoiced"]
df_valid_orders = df_customer_spending[df_customer_spending["order_status"].isin(valid_statuses)]

# Aggregate total spending per unique customer identifier
df_clv_base = df_valid_orders.groupby("customer_unique_id")["payment_value"].sum().reset_index()
df_clv_base.rename(columns={"payment_value": "total_lifetime_spend"}, inplace=True)

# Re-introduce master customer index to preserve unfulfilled accounts
df_all_humans = df_customers[["customer_unique_id"]].drop_duplicates()
df_clv = pd.merge(df_all_humans, df_clv_base, on="customer_unique_id", how="left")

# Apply sentinel value to flag unfulfilled or unpaid order flows
df_clv["total_lifetime_spend"] = df_clv["total_lifetime_spend"].fillna(-1)

# Categorize customer base into target lifetime value brackets
def assign_clv_tier(spend):
    if spend == -1:
        return "4. Lost Revenue (Canceled / Unavailable)"
    elif spend >= 500:
        return "1. VIP Spender ($500+)"
    elif spend >= 100:
        return "2. Regular Customer ($100-$499)"
    else:
        return "3. Low Spend / Trial (<$100)"

df_clv["clv_tier"] = df_clv["total_lifetime_spend"].apply(assign_clv_tier)
df_clv["total_lifetime_spend"] = df_clv["total_lifetime_spend"].replace(-1, 0)

# Generate baseline frequency distribution report
clv_distribution = df_clv["clv_tier"].value_counts().sort_index().reset_index()
clv_distribution.columns = ["CLV Tier", "Total Unique Customers"]
total_customers = clv_distribution["Total Unique Customers"].sum()
clv_distribution["Percentage of Base"] = (clv_distribution["Total Unique Customers"] / total_customers * 100).round(2).astype(str) + "%"

# Export clean customer dimension table for Power BI ingestion
df_clv.to_csv("../data_raw/clean_customer_clv.csv", index=False)

print("'clean_customer_clv.csv' saved safely inside data_raw folder!")
print(f"Total Profiles Mapped: {total_customers}\n")
clv_distribution

Processing Advanced 4-Tier CLV Lifecycle Engine...
'clean_customer_clv.csv' saved safely inside data_raw folder!
Total Profiles Mapped: 96096



,CLV Tier,Total Unique Customers,Percentage of Base
0,1. VIP Spender ($500+),4384,4.56%
1,2. Regular Customer ($100-$499),46667,48.56%
2,3. Low Spend / Trial (<$100),43935,45.72%
3,4. Lost Revenue (Canceled / Unavailable),1110,1.16%


In [6]:
# Filter and validate entries lacking explicit English translations
unclassified_rows = df_clean_products[df_clean_products["product_category_name_english"] == "Unclassified/Unknown"]
print(f"Total unclassified inventory records: {len(unclassified_rows)}")

unclassified_rows[["product_id", "product_category_name", "product_category_name_english"]].head(20)

Total unclassified inventory records: 623


,product_id,product_category_name,product_category_name_english
105,a41e356c76fab66334f36de622ecbd3a,unclassified,Unclassified/Unknown
128,d8dee61c2034d6d075997acef1870e9b,unclassified,Unclassified/Unknown
145,56139431d72cd51f19eb9f7dae4d1617,unclassified,Unclassified/Unknown
154,46b48281eb6d663ced748f324108c733,unclassified,Unclassified/Unknown
197,5fb61f482620cb672f5e586bb132eae9,unclassified,Unclassified/Unknown
244,e10758160da97891c2fdcbc35f0f031d,unclassified,Unclassified/Unknown
294,39e3b9b12cd0bf8ee681bbc1c130feb5,unclassified,Unclassified/Unknown
299,794de06c32a626a5692ff50e4985d36f,unclassified,Unclassified/Unknown
347,7af3e2da474486a3519b0cba9dea8ad9,unclassified,Unclassified/Unknown
428,629beb8e7317703dcc5f35b5463fd20e,unclassified,Unclassified/Unknown
